# 3. Spark

Spark Programming Guide: <https://spark.apache.org/docs/latest/> (use Python API recommended)
Spark API: <https://spark.apache.org/docs/latest/api/python/index.html>


# 3.1 Example Walkthrough
Follow the Spark examples below. After completion, see Exercise 3.2 and 3.3 at the bottom of this notebook.


### Initialize PySpark

In [1]:
!pip install pyspark

In [2]:
import os, sys

In [3]:
# Initialize PySpark
APP_NAME = "PySpark Lecture"
SPARK_MASTER="local[1]"
import pyspark
import pyspark.sql
from pyspark.sql import Row
conf=pyspark.SparkConf()
conf=pyspark.SparkConf().setAppName(APP_NAME).set("spark.local.dir", os.path.join(os.getcwd(), "tmp"))
sc = pyspark.SparkContext(master=SPARK_MASTER, conf=conf)
spark = pyspark.sql.SparkSession(sc).builder.appName(APP_NAME).getOrCreate()

print("PySpark initiated...")

PySpark initiated...


### Hello, World!

Loading data, mapping it and collecting the records into RAM...

In [4]:
!wget https://raw.githubusercontent.com/scalable-infrastructure/exercise-2026/main/data/example.csv

--2026-04-03 08:08:28--  https://raw.githubusercontent.com/scalable-infrastructure/exercise-2026/main/data/example.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 189 [text/plain]
Saving to: ‘example.csv’

example.csv         100%[===================>]     189  --.-KB/s    in 0s      

2026-04-03 08:08:28 (3.11 MB/s) - ‘example.csv’ saved [189/189]



In [5]:
# Load the text file using the SparkContext
csv_lines = sc.textFile("example.csv")

# Map the data to split the lines into a list
data = csv_lines.map(lambda line: line.split(","))

# Collect the dataset into local RAM
data.collect()

[['Russell Jurney', 'Relato', 'CEO'],
 ['Florian Liebert', 'Mesosphere', 'CEO'],
 ['Don Brown', 'Rocana', 'CIO'],
 ['Steve Jobs', 'Apple', 'CEO'],
 ['Donald Trump', 'The Trump Organization', 'CEO'],
 ['Russell Jurney', 'Data Syndrome', 'Principal Consultant']]

### Creating Objects from CSV

Using a function with a map operation to create objects (dicts) as records...

In [6]:
# Turn the CSV lines into objects
def csv_to_record(line):
    parts = line.split(",")
    record = {
      "name": parts[0],
      "company": parts[1],
      "title": parts[2]
    }
    return record

# Apply the function to every record
records = csv_lines.map(csv_to_record)

# Inspect the first item in the dataset
records.first()

{'name': 'Russell Jurney', 'company': 'Relato', 'title': 'CEO'}

### GroupBy

Using the groupBy operator to count the number of jobs per person...

In [7]:
# Group the records by the name of the person
grouped_records = records.groupBy(lambda x: x["name"])

# Show the first group
grouped_records.first()

# Count the groups
job_counts = grouped_records.map(
  lambda x: {
    "name": x[0],
    "job_count": len(x[1])
  }
)

job_counts.first()

job_counts.collect()

[{'name': 'Russell Jurney', 'job_count': 2},
 {'name': 'Florian Liebert', 'job_count': 1},
 {'name': 'Don Brown', 'job_count': 1},
 {'name': 'Steve Jobs', 'job_count': 1},
 {'name': 'Donald Trump', 'job_count': 1}]

### Map vs FlatMap

Understanding the difference between the map and flatmap operators...

In [8]:
# Compute a relation of words by line
words_by_line = csv_lines\
  .map(lambda line: line.split(","))

print(words_by_line.collect())

# Compute a relation of words
flattened_words = csv_lines\
  .map(lambda line: line.split(","))\
  .flatMap(lambda x: x)

flattened_words.collect()

[['Russell Jurney', 'Relato', 'CEO'], ['Florian Liebert', 'Mesosphere', 'CEO'], ['Don Brown', 'Rocana', 'CIO'], ['Steve Jobs', 'Apple', 'CEO'], ['Donald Trump', 'The Trump Organization', 'CEO'], ['Russell Jurney', 'Data Syndrome', 'Principal Consultant']]


['Russell Jurney',
 'Relato',
 'CEO',
 'Florian Liebert',
 'Mesosphere',
 'CEO',
 'Don Brown',
 'Rocana',
 'CIO',
 'Steve Jobs',
 'Apple',
 'CEO',
 'Donald Trump',
 'The Trump Organization',
 'CEO',
 'Russell Jurney',
 'Data Syndrome',
 'Principal Consultant']

---
## Further Exercises




In [9]:
!wget "https://github.com/scalable-infrastructure/exercise-2026/blob/main/data/nasa/NASA_access_log_Jul95.gz?raw=true" -O NASA_access_log_Jul95.gz
!gzip -d NASA_access_log_Jul95.gz

--2026-04-03 08:08:37--  https://github.com/scalable-infrastructure/exercise-2026/blob/main/data/nasa/NASA_access_log_Jul95.gz?raw=true
Resolving github.com (github.com)... 140.82.114.3
Connecting to github.com (github.com)|140.82.114.3|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://github.com/scalable-infrastructure/exercise-2026/raw/refs/heads/main/data/nasa/NASA_access_log_Jul95.gz [following]
--2026-04-03 08:08:38--  https://github.com/scalable-infrastructure/exercise-2026/raw/refs/heads/main/data/nasa/NASA_access_log_Jul95.gz
Reusing existing connection to github.com:443.
HTTP request sent, awaiting response... 302 Found
Location: https://raw.githubusercontent.com/scalable-infrastructure/exercise-2026/refs/heads/main/data/nasa/NASA_access_log_Jul95.gz [following]
--2026-04-03 08:08:38--  https://raw.githubusercontent.com/scalable-infrastructure/exercise-2026/refs/heads/main/data/nasa/NASA_access_log_Jul95.gz
Resolving raw.githubusercontent.c

## 3.2 Word Count

Implement a word count using Spark. How many words are in the file `example.csv`?

In [10]:
## 3.3 HTTP Response Code Analysis

Using the NASA log file, implement a Spark version of the HTTP Response Code Analysis. How many log entries per HTTP Response Code exist?

Object `exist` not found.


In [11]:
#A3.2: Word Count

word_frequencies = csv_lines.flatMap(lambda line: line.split(",")) \
                            .map(lambda word: (word, 1)) \
                            .reduceByKey(lambda a, b: a + b)

print("--- Word Frequencies ---")
print(word_frequencies.collect())

total_words = csv_lines.flatMap(lambda line: line.split(",")).count()

print(f"\nTotal number of words in example.csv: {total_words}")

--- Word Frequencies ---
[('Russell Jurney', 2), ('Relato', 1), ('CEO', 4), ('Florian Liebert', 1), ('Mesosphere', 1), ('Don Brown', 1), ('Rocana', 1), ('CIO', 1), ('Steve Jobs', 1), ('Apple', 1), ('Donald Trump', 1), ('The Trump Organization', 1), ('Data Syndrome', 1), ('Principal Consultant', 1)]

Total number of words in example.csv: 18


In [12]:
#A3.3: HTTP Response Code Analysis

nasa_lines = sc.textFile("NASA_access_log_Jul95")

def extract_status_code(line):
    parts = line.split()
    if len(parts) >= 2:
        return parts[-2]
    return "UNKNOWN"

response_code_counts = nasa_lines.filter(lambda line: len(line.strip()) > 0) \
                                 .map(extract_status_code) \
                                 .map(lambda code: (code, 1)) \
                                 .reduceByKey(lambda a, b: a + b) \
                                 .sortBy(lambda x: x[1], ascending=False)

print("--- Log entries per HTTP Response Code ---")
for code, count in response_code_counts.collect():
    print(f"HTTP {code}: {count} entries")

--- Log entries per HTTP Response Code ---
HTTP 200: 1701534 entries
HTTP 304: 132627 entries
HTTP 302: 46573 entries
HTTP 404: 10845 entries
HTTP 500: 62 entries
HTTP 403: 54 entries
HTTP 501: 14 entries
HTTP 400: 5 entries
HTTP UNKNOWN: 1 entries
